# Evaluation - Charge-Light Matching - AFTER Beam Window Cut

Same multi-file evaluation as
`Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb`, with the
**beam-window cut applied to the reco side**: only clustering-global clusters
whose bridged flash time falls inside
`[BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]` = [0.33, 1.93] us survive into
efficiency, purity, matching, metadata and every plot. The in-spill reco
population is neutrino-dominated, so this is the notebook for questions about
how well **neutrino** clusters are reconstructed.

The cut is deliberately reco-side only. "In beam window" is not a truth
quantity -- true clusters carry no flash and no time -- so a true-side version
could only be inferred by matching to a beam-window-flashed reco cluster,
folding beam timing into what would read as a truth-level selection. The true
side therefore still contains its cosmic clusters; a cosmic true cluster that
now goes unmatched means "no in-spill reco cluster near it", which is the
intended reading rather than a bug.

Set `Apply_beam_window_cut = False` in the configuration cell to reproduce the
before-cut notebook exactly. Output goes to
`multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut/`, a
subdirectory of the shared charge-light tree, so the two notebooks never
overwrite each other.

---


Evaluation for the **charge-light matching** JSON file format (combined-APA:
`img-global` reco clusters, `sed-sce_drift_smear_readout` true clusters, `mc`
particle truth tree, `op` optical/light info). This is a new, separate JSON
format/layout from the one `Evaluation_BeforeChargeLightMatching_BeforeBeamWindowCut.ipynb` reads —
that notebook and the modules it calls (`readfiles.py`, `selections.py`,
`efficiency_purity_estimate.py`, `efficiency_purity_draw.py`, etc.) are left
completely unchanged; the charge-light readers were added additively
alongside the existing ones in `readfiles.py`.

**Current stage**: this notebook unzips the test data (once) and reads the
four charge-light files above for each file/event, printing sanity-check
counts. The downstream selections / efficiency / purity / drawing pipeline
is not wired in yet — that comes once the cluster-ID / neutrino-identification
mapping for this format is defined.

In [1]:
# Charge-light matching is a combined-APA evaluation (img-global / sed-sce are
# already global across APAs) -- no per-APA/face looping like the older pipeline.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None   # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

# Range of event numbers to run, inclusive on both ends: (1, 5) runs events
# 1,2,3,4,5. None runs every event. Applied on top of target_event, so leave
# target_event = None when using a range. This exists because target_event only
# ever compares equal to one int -- handing it a tuple silently matched nothing
# and produced a run with zero events and no plots.
target_event_range = None   # e.g. (1, 5) for events 1..5

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. This is the only file selector that survives
# that sort order; the `files = N` knob above still takes the first N in
# lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9

# Fail fast rather than silently processing nothing: `evt != target_event` can
# never be False for a tuple, so target_event = (1, 5) used to skip every event
# and still write an empty summary.
if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")


# ========================================================================
# Decide which plots to draw
# ========================================================================
b_draw_event_level_plots = True   # Draw event-level plots (one per event)
b_draw_file_level_plots  = True   # Draw file-level plots (one per file)
b_draw_job_level_plots   = True    # Draw job-level plots (one per job)

In [2]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
import os
from scipy.spatial import KDTree
import pandas as pd
import seaborn as sns
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Job started at: 2026-08-03 14:38:12


In [3]:
# Import functions from Python modules
# Full pipeline now wired in: selections, cluster_category, efficiency, purity,
# 1-to-1/1-to-many matching, metadata, and all drawing. reassign_cluster_ID_true_charge_light
# IS used -- true clusters are grouped under 99990+nu_idx (99991, 99992, ... one
# per neutrino interaction) / avg-X (cosmic); see that function's docstring in
# selections.py for why this is a separate function from the legacy
# reassign_cluster_ID_true (still 9999-merging, still used by other pipelines).
# reassign_cluster_ID_reco IS ALSO used -- reco (clustering-global) clusters
# are relabeled by avg-X, same convention.
from readfiles import ensure_data_extracted, read_charge_light_files_for_event, flatten_mc_tree
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light, reassign_cluster_ID_reco,
    apply_energy_cutoff, apply_min_true_points_cutoff, apply_min_reco_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
)
from cluster_category import cluster_category
from efficiency_purity_estimate import EvaluateEfficiency, EvaluatePurity
from clusterpairmatching import MatchTrueToReco1to1, MatchTruetoReco_OneToMany
from metadata import (
    build_cluster_flash_metadata, build_img_cluster_flash_metadata, CATHODE_CROSSING_TIME_DIFF_MAX_US,
    add_metadata_true_clusters, add_metadata_true_reco_pair_cluster,
    build_true_cluster_type_records, build_neutrino_vertex_records,
)
# Information-file writers live in writeinformation.py, not metadata.py: metadata
# builds the in-memory records, this writes the human-readable .txt tables.
# write_neutrino_vertex_info / write_removed_neutrino_info are imported again: the
# job-level true_neutrino_info.txt and removed_true_neutrino_info.txt are written
# HERE as well as in SelectionAnalysis.ipynb. Same writers, same filenames, fed the
# same records -- this notebook's copies describe the population its own efficiency
# and purity numbers were computed on, so the two are read together rather than
# having to cross-reference the other notebook's output tree.
from writeinformation import (
    write_true_cluster_info, write_reco_cluster_info,
    write_neutrino_vertex_info, write_removed_neutrino_info,
    write_efficiency_info, write_pair_efficiency_info, write_purity_info,
)
from efficiency_purity_draw import (
    plot_efficiency_heatmap, plot_purity_heatmap,
    DrawEfficiencyVsTrueEnergyPerEvent, DrawEfficiencyVsTrueEnergyPerFile, DrawEfficiencyVsTrueEnergyPerJob,
    DrawClusterEfficiencyVsTrueEnergyPerEvent, DrawClusterEfficiencyVsTrueEnergyPerFile, DrawClusterEfficiencyVsTrueEnergyPerJob,
    DrawEfficiencyVsTrueEnergy_MatchedPairs_PerEvent, DrawEfficiencyVsTrueEnergy_MatchedPairs_PerFile, DrawEfficiencyVsTrueEnergy_MatchedPairs_PerJob,
    DrawPurityVsRecoChargePerEvent, DrawPurityVsRecoChargePerFile, DrawPurityVsRecoChargePerJob,
    DrawEfficiencyVsPurity_MatchedPairs,
    summarize_cluster_efficiency_by_energy, format_cluster_efficiency_by_energy,
)
from DrawRecoTrueClusters import (
    DrawTrueRecoClustersXZ, DrawTrueRecoClustersYZ, DrawTrueRecoClustersXY, DrawLabels, DrawNeutrinoRecoClusters,
    DrawTrueClusterWithMatchedReco, DrawTrueRecoMatchMultiplicity, DrawLabelPerFile,
    draw_clustering_global_clusters, DrawLabelsAggregated, DrawLabelsByNuIdx,
    DrawNeutrinoVertices, DrawNeutrinoVolumeCategory, DrawNeutrinoFlavor, DrawNeutrinoBreakdown,
    _draw_cosmic_category_bar,
)
from DrawRecoTrueFlashes import (
    draw_flashes, draw_img_global_clusters, draw_clustering_flashes, draw_cluster_flash_time_bar,
    BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US,
)


In [4]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-img-global.json                      (reco clusters, combined APA)
#   file0/data/0/0-sed-sce_drift_smear_readout.json      (true clusters, combined APA)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/0/0-op.json                               (optical/light info)
#   file0/data/1/, 2/, ... (one subdirectory per event)
#   file1/mabc.zip, file1/data/..., etc.

# The DEAD-AREA-PREPROCESSED tree, produced once by preprocess_deadarea_cut.py.
# Its true-point files already have the dead-area cut applied, which is why
# Apply_deadarea_cut is False below -- see the note there. Read that script's
# docstring, or DEADAREA_PREPROCESSING.txt inside the tree, before switching this
# back to the raw tree: the two are NOT interchangeable.
PARENT_DIR = Path("Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut")

# Number of files to process (convert 'files' variable to num_files_to_process)
num_files_to_process = None if files == "all" else files

# Number of events to process (convert 'events' variable to num_events_to_process)
num_events_to_process = None if events == "all" else events

# Output directory for plots: a subdirectory of the shared charge-light tree, NOT
# the tree root. The before-cut notebook writes its filenames straight into the
# root, so sharing that level would have whichever notebook ran last silently
# overwrite the other's plots and summaries.
PLOTBASEDIR = Path("multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS
# ========================================================================
# Matching/efficiency/purity radii and the geometry-based cuts (fiducial YZ box,
# dead-area) are reused at the SAME values as the existing pipeline -- detector
# geometry hasn't changed and these are unit-independent of the new q/energy field.
radius_efficiency         = 2
radius_purity_xz          = 2
radius_purity_yz          = 5
radius_purity_xy          = 5
min_recopoints_threshold  = 5

# min_true_points_cutoff / min_reco_points_cutoff are DISABLED for now: this
# format's point clouds are much sparser than the old imaging-based
# reconstruction -- real neutrino clusters have been seen with as few as 13
# points -- so the old threshold (200) would delete real signal clusters
# outright. Revisit once correct values are known for this format's point
# density.
#
# min_cluster_energy IS applied (Apply_energy_cutoff = True): sed-sce's
# per-point 'e' field (MeV) is a genuine energy deposit -- same physical
# quantity/units as the old (non charge-light) pipeline's energy column --
# so the old threshold (100 MeV) carries over directly. See
# build_true_points_charge_light's energy= parameter (falls back to 'q'
# for older-format files that lack 'e').
min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)
min_reco_points_cutoff    = 200     # NOT APPLIED (Apply_min_reco_points_cutoff = False below)

# Apply selections
Apply_energy_cutoff                         = True    # sed-sce's 'e' field is genuine MeV -- see note above
Apply_min_true_points_cutoff                = False   # disabled -- see note above
Apply_min_reco_points_cutoff                = False   # disabled -- see note above
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# BEAM-WINDOW (time) CUT -- the one cut that makes this the "after beam window
# cut" notebook. Keeps only reco clusters whose bridged flash time lies inside
# [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US] = [0.33, 1.93] us, i.e. the in-spill
# population, which is neutrino-dominated. Set False to reproduce
# Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb exactly.
#
# This is NOT the same cut as Apply_time_window_cut above: that one needs a
# per-point TRUE time, which this format does not carry. The beam window is
# measured where the timing actually lives -- on the reco side, via the flash.
Apply_beam_window_cut                       = True
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Applying it again would be a no-op that
# costs a polygon test per event. It is applied FIRST, before the energy cut,
# because reco points are never reconstructed inside a dead channel region -- true
# deposits there could never have been seen, so removing them is a correction that
# puts truth and reco on the same measurable volume, not a selection to rank
# alongside the others. Set this True only if you point PARENT_DIR back at a raw
# tree, and note that doing so restores the OLD ordering (dead area last).
Apply_deadarea_cut                          = False

# YZ Sensitivity cut parameters (detector geometry, unit: cm)
x_min = -250.0
x_max = 250.0
y_min = -200.0
y_max = 200.0
z_min = 0.15
z_max = 500.85

# Metadata label only (add_metadata_true_clusters/add_metadata_true_reco_pair_cluster
# store 'view' as a plain string field, not used for any logic) -- there's no
# 2-view/3-view distinction in the charge-light format, so this is just a constant.
view = "combined"

marker_size = 1

print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied (threshold {min_cluster_energy} MeV, using sed-sce's per-point 'e' field)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_beam_window_cut:
    print(f"- Beam window cut applied to RECO clusters only ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us flash time)")
else:
    print(f"- Beam window cut NOT applied -- this run reproduces the before-beam-window-cut notebook")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE (split by X sign into APA0/APA1, see apply_deadarea_cut_true_charge_light)")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR, before every other cut)")
print("\nCuts not applied (see note above)")
if not Apply_min_true_points_cutoff:
    print(f"- Minimum true points cutoff not applied (threshold {min_true_points_cutoff} too aggressive for this format's point density)")
if not Apply_min_reco_points_cutoff:
    print(f"- Minimum reco points cutoff not applied (threshold {min_reco_points_cutoff} too aggressive for this format's point density)")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# Each file subdirectory ships as a single zip file. ensure_data_extracted()
# only unzips if that file's data/ folder doesn't already exist yet, so
# re-running this notebook never re-extracts.
if PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


Configuration:
Parent directory: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
Plot base directory: multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut
Files to process: all
Events to process: all

Cuts applied:
- Energy cutoff applied (threshold 100 MeV, using sed-sce's per-point 'e' field)
- Wire readout sensitive xz plane cut applied
- Beam window cut applied to RECO clusters only (0.33 - 1.93 us flash time)
- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR, before every other cut)

Cuts not applied (see note above)
- Minimum true points cutoff not applied (threshold 200 too aggressive for this format's point density)
- Minimum reco points cutoff not applied (threshold 200 too aggressive for this format's point density)


In [5]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs

def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                event_num = int(item.name)
                events.append(event_num)
            except ValueError:
                pass

    return sorted(events)

# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# Limit to num_files_to_process files for testing
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


Scanning parent directory: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
------------------------------------------------------------
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file1
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file10
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file11
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file2
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file3
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file4
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file5
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file6
Fou

In [6]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA charge-light matching evaluation
# ============================================================================
# CLUSTERING-LEVEL evaluation (after charge-light-matching, CLM): selections /
# cluster_category / efficiency / purity / 1-to-1 & 1-to-many matching /
# metadata / drawing are all computed on sed-smear_readout (true, no SCE
# correction) vs clustering-global (reco, post-CLM) -- NOT img-global/sed-sce
# (imaging-level, pre-CLM), which are still read (for the flash-bridging
# logic below) but no longer feed the efficiency/purity/matching pipeline.
# reassign_cluster_ID_true_charge_light IS used (true clusters grouped under
# 99990+nu_idx -- 99991, 99992, ... one per neutrino interaction -- instead
# of a single shared 9999, so multiple neutrino interactions in the same
# event are no longer merged into one true cluster; avg-X for cosmic,
# unchanged). See selections.py for why this is a separate function from the
# legacy reassign_cluster_ID_true (still 9999-merging, still used by other
# pipelines). reassign_cluster_ID_reco IS ALSO used (reco clusters relabeled
# by avg-X) -- but only for the efficiency/purity/matching pipeline's
# clusters_reco; the earlier clusters_all_clu (flash-time bridging /
# beam-window highlighting) stays keyed by clustering-global's ORIGINAL
# cluster_id, since that's the namespace build_img_cluster_flash_metadata's
# records reference.
#
# IMPORTANT: clustering-global points are grouped by REAL_CLUSTER_ID, not
# cluster_id -- clustering-global's own 'cluster_id' is a COARSER grouping
# that can merge multiple physically distinct tracks together (confirmed
# against real data via BEE display comparison: a single cluster_id spanning
# two disjoint Y ranges that split cleanly into two real_cluster_id values,
# each matching a different true cluster). 'real_cluster_id' is the
# physically correct per-track ID. (img-global does NOT have this issue --
# cluster_id == real_cluster_id everywhere there -- so img-global-side
# grouping is unaffected and still uses cluster_id.)
#
# q_true is now taken from sed-smear's 'nu_idx' field when available (0=cosmic,
# 1/2/...=which neutrino interaction), not just a binary 0/1 flag -- see
# build_true_points_charge_light's nu_idx= parameter. Combined with
# reassign_cluster_ID_true_charge_light's 99990+nu_idx scheme above, each
# neutrino interaction is now its own true cluster, so DrawLabelsAggregated's
# "Neutrino" cluster count and the sum of DrawLabelsByNuIdx's per-nu_idx bars
# now agree (previously they diverged, since all neutrino interactions were
# merged under one cluster_id=9999 and DrawLabelsByNuIdx counted nu_idx
# values found inside that single merged cluster). NOTE: apply_energy_cutoff
# / apply_min_true_points_cutoff run AFTER reassignment, so with this scheme
# each neutrino interaction's energy/point-count is now cut independently,
# rather than pooled together as before. The pre-existing
# DrawLabels/DrawLabelPerFile calls stay as-is (unmodified, additive-only)
# but their strict q_true==1 point-level check still only recognizes
# nu_idx=1 as neutrino, undercounting nu_idx=2/3 clusters -- unaffected by
# this ID scheme change, since that check never looked at cluster_id. At job
# level, DrawLabelPerJob itself is no longer called (its neutrino-vs-cosmic
# bar, true_clusters_by_type_job_*events_Combined.png, isn't wanted) -- only
# its cosmic-category-breakdown half is still drawn, via
# _draw_cosmic_category_bar directly (same helper DrawLabelPerJob itself
# calls internally), so cosmic_clusters_by_category_job_*events_Combined.png
# is unaffected.
#
# Note on the "imaginglevel"/"clusteringlevel" directory names below: since
# EVERYTHING here is post-CLM data now, these names no longer distinguish
# data source (that's what "afterCLM" used to flag, now dropped as redundant)
# -- they instead distinguish COMPUTATION STYLE, echoing the original
# pipeline's naming: "imaginglevel" = direct per-true-cluster efficiency
# (DrawEfficiencyVsTrueEnergyPerEvent, summed across all reco matches, no
# purity); "clusteringlevel" = 1-to-1 best-match pairing
# (DrawClusterEfficiencyVsTrueEnergyPerEvent / MatchedPairs variant, with
# purity attached).
#
# BEAM_WINDOW_MIN_US / BEAM_WINDOW_MAX_US imported from DrawRecoTrueFlashes.

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

job_efficiency_results          = []
job_purity_results              = []
job_metadata_list               = []      # per-true-cluster metadata (add_metadata_true_clusters)
job_pair_metadata_list          = []      # per 1-to-1 true-reco pair metadata (add_metadata_true_reco_pair_cluster)
job_flash_metadata_list         = []      # per-cluster flash records (build_cluster_flash_metadata)
job_img_cluster_flash_records   = []      # per-clustering-cluster flash records (build_img_cluster_flash_metadata)
job_cluster_type_records        = []     # per-true-cluster is_neutrino records (build_true_cluster_type_records)
job_vertex_records              = []      # per true neutrino interaction (build_neutrino_vertex_records)
job_reco_beam_window_records    = []      # per-event count of clustering-global clusters with a beam-window flash
total_events_processed          = 0
total_files_processed           = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    # SELECTIVE FILTERING: Skip files outside target_file_range (inclusive both
    # ends, matched on the directory name's trailing index -- see
    # file_index_from_name). Applied on top of target_file, not instead of it.
    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    file_output_dir = output_dir / input_file_name
    file_output_dir.mkdir(parents=True, exist_ok=True)

    # Containers for file-level aggregation
    file_efficiency_results         = []
    file_purity_results             = []
    file_metadata_list              = []
    file_pair_metadata_list         = []
    file_flash_metadata_list        = []
    file_img_cluster_flash_records  = []
    file_cluster_type_records       = []
    file_reco_beam_window_records   = []
    file_vertex_records             = []      # per true neutrino interaction (build_neutrino_vertex_records)

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    if num_events_to_process is None:
        event_high = max(events_list) + 1  # Process all events
    else:
        event_high = event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1


    # Start of event loop
    for evt in range(event_low, event_high):
        # SELECTIVE FILTERING: Skip events that don't match target_event
        if target_event is not None and evt != target_event:
            continue

        # SELECTIVE FILTERING: Skip events outside target_event_range (inclusive
        # both ends). Applied on top of target_event, not instead of it.
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key = f"{input_file_name}_{evt}"
        event_output_dir = file_output_dir / f"event_{evt:03d}"
        event_output_dir.mkdir(parents=True, exist_ok=True)
        PLOTDIR_EVT = event_output_dir

        efficiency_dir = event_output_dir / "efficiency"
        purity_dir     = event_output_dir / "purity"
        efficiency_dir.mkdir(parents=True, exist_ok=True)
        purity_dir.mkdir(parents=True, exist_ok=True)

        # Sub-directories for 2D/1D efficiency-vs-true-energy plots, split by computation
        # style (see header note above): "imaginglevel" (every true cluster, summed
        # efficiency across all reco matches, no purity), "clusteringlevel" (1-to-1 best
        # match + unmatched true clusters at efficiency=0, with purity),
        # "clusteringlevel_true_reco_pairs_only" (1-to-1 matched pairs only, no unmatched).
        efficiency_2d1d_imaging_dir                = efficiency_dir / "efficiency_2d_1d_imaginglevel"
        efficiency_2d1d_clustering_dir             = efficiency_dir / "efficiency_2d_1d_clusteringlevel"
        efficiency_2d1d_clustering_pairs_only_dir  = efficiency_dir / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
        efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
        efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
        efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

        # Sub-directories for efficiency-vs-purity matched-pair plots: one including
        # unmatched true clusters (drawn in a dedicated "no match" box), one excluding them
        eff_vs_purity_incl_dir = event_output_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
        eff_vs_purity_excl_dir = event_output_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
        eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
        eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

        # Sub-directories for imaging-level flash/cluster diagnostics
        imaging_details_flashes_dir  = event_output_dir / "imaging_details" / "flashes"
        imaging_details_clusters_dir = event_output_dir / "imaging_details" / "clusters"
        imaging_details_flashes_dir .mkdir(parents=True, exist_ok=True)
        imaging_details_clusters_dir.mkdir(parents=True, exist_ok=True)

        # Sub-directories for clustering-level (post charge-light-matching) diagnostics
        clustering_details_flashes_dir  = event_output_dir / "clustering_details" / "flashes"
        clustering_details_clusters_dir = event_output_dir / "clustering_details" / "clusters"
        clustering_details_flashes_dir .mkdir(parents=True, exist_ok=True)
        clustering_details_clusters_dir.mkdir(parents=True, exist_ok=True)

        x_reco, y_reco, z_reco, id_reco, q_reco, real_id_reco = result['reco']
        # sed-smear_readout (NOT sed-sce) -- clustering-level (post-CLM) truth, paired
        # with clustering-global's reco below. sed-sce (imaging-level, pre-CLM truth,
        # result['true']) is still returned by read_charge_light_files_for_event but
        # no longer feeds this pipeline.
        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result['true_clustering']
        mc_tree = result['mc']
        op_data = result['op']

        # ------------------------------------------------------------------
        # FLASH METADATA (op.json)
        # ------------------------------------------------------------------
        event_flash_metadata_list = build_cluster_flash_metadata(op_data, input_file_name, evt, "Combined", event_key)
        if b_draw_event_level_plots:
            draw_flashes(event_flash_metadata_list, imaging_details_flashes_dir, "Combined", f"Event {evt}", f"event_{evt}", file_name=input_file_name, write_text_table=True)  

        # ------------------------------------------------------------------
        # IMG-GLOBAL CLUSTERS (event level only): grouped by the RAW
        # img-global cluster_id (point-wise, matched by array index) --
        # NOT reassigned via reassign_cluster_ID_reco, since that's the
        # namespace op_cluster_ids uses to reference clusters. (img-global's
        # cluster_id == real_cluster_id everywhere, so no distinction here.)
        # ------------------------------------------------------------------
        predicted_points_raw = np.column_stack((x_reco, y_reco, z_reco, id_reco, q_reco))
        clusters_all_raw = GroupClustersByID(predicted_points_raw)

        flash_matched_ids = {float(r['reco_cluster_id']) for r in event_flash_metadata_list}
        beam_window_ids   = {float(r['reco_cluster_id']) for r in event_flash_metadata_list
                              if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}

        clusters_with_flash      = {cid: pts for cid, pts in clusters_all_raw.items() if cid in flash_matched_ids}
        clusters_in_beam_window  = {cid: pts for cid, pts in clusters_all_raw.items() if cid in beam_window_ids}

        if b_draw_event_level_plots:
            draw_img_global_clusters(clusters_all_raw, clusters_with_flash, clusters_in_beam_window,
                                     evt, "Combined", imaging_details_clusters_dir, file_name=input_file_name)

        # ------------------------------------------------------------------
        # CLUSTERING-GLOBAL <-> IMG-GLOBAL FLASH BRIDGE (event level for the
        # spatial/bar plots; the num-clusters-vs-flash-time plot also
        # aggregates to file/job level below). clustering-global's cluster_id
        # is a different numbering scheme than img-global's -- the bridge is
        # by point-level charge ('q') value, see build_img_cluster_flash_metadata.
        # clusters_all_clu here is grouped by clustering-global's
        # REAL_CLUSTER_ID (NOT cluster_id, which merges distinct tracks --
        # see header note; also NOT reassign_cluster_ID_reco'd) since
        # real_cluster_id is the namespace build_img_cluster_flash_metadata's
        # records now reference.
        # ------------------------------------------------------------------
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)

        if b_draw_event_level_plots:
            draw_clustering_flashes(event_img_cluster_flash_records, clustering_details_flashes_dir, "Combined",
                                     f"Event {evt}", f"event_{evt}", file_name=input_file_name, write_text_table=True)
            draw_cluster_flash_time_bar(event_img_cluster_flash_records, evt, "Combined",
                                         clustering_details_flashes_dir, file_name=input_file_name)

        x_clu, y_clu, z_clu, id_clu, q_clu, real_id_clu = result['clustering']
        predicted_points_clu = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))
        clusters_all_clu = GroupClustersByID(predicted_points_clu)

        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                                if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        clusters_clu_in_beam_window = {cid: pts for cid, pts in clusters_all_clu.items() if cid in clu_beam_window_ids}

        # Per-event reco-side beam-window record (job level: reco_cluster_info.txt) --
        # counts DISTINCT clustering-global clusters with a beam-window-matched flash,
        # independent of any true-cluster info: a proxy for multiple neutrino-like
        # activity in the beam spill from the RECO side (unlike true_cluster_info.txt's
        # num_neutrinos, which is ground truth via nu_idx).
        event_reco_beam_window_record = {
            'file_name': input_file_name,
            'event': event_key,
            'event_num': evt,
            'num_clusters_in_beam_window': len(clusters_clu_in_beam_window),
            'cluster_ids': sorted(clusters_clu_in_beam_window.keys()),
        }


        if b_draw_event_level_plots:
            draw_clustering_global_clusters(clusters_all_clu, clusters_clu_in_beam_window,
                                             evt, "Combined", clustering_details_clusters_dir, file_name=input_file_name)

        # ------------------------------------------------------------------
        # TRUE POINTS: adapt sed-smear_readout into the standard 7-column
        # shape (energy column is sed-smear's per-point 'e' field, genuine
        # MeV; q_true column is sed-smear's own 'nu_idx' field when available
        # -- 0=cosmic, 1/2/...=which neutrino interaction), reassign IDs
        # (99990+nu_idx for neutrino -- one cluster per interaction -- avg-X
        # for cosmic), then apply the same selection cuts as the existing
        # pipeline.
        # ------------------------------------------------------------------
        # real_id_true (real_cluster_id), NOT id_true (cluster_id): same reason as the
        # reco side -- clustering-global's cluster_id is a coarser grouping that can
        # merge physically distinct tracks together, real_cluster_id is the physically
        # correct per-track ID. The two arrays are identical in every sed-smear file in
        # the current tree, so this changes no present result -- it makes the true side
        # robust the way the reco side already is.
        true_points = build_true_points_charge_light(x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points = reassign_cluster_ID_true_charge_light(true_points)

        # Snapshot BEFORE the cuts: build_neutrino_vertex_records uses it to say
        # what a removed neutrino actually deposited, and hence which cut removed
        # it (removed_true_neutrino_info.txt). Grouping only, no filtering.
        clusters_true_precut = GroupClustersByID(true_points)

        if Apply_energy_cutoff:
            true_points = apply_energy_cutoff(true_points, min_cluster_energy)
        if Apply_min_true_points_cutoff:
            true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            true_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_points, x_min, x_max, y_min, y_max, z_min, z_max)
        if Apply_deadarea_cut:
            true_points = apply_deadarea_cut_true_charge_light(true_points, output_dir=PLOTDIR_EVT, event=evt, file_name=input_file_name)

        if len(true_points) == 0:
            print(f"  Event {evt}: no true points remain after cuts, skipping")
            continue

        clusters_true = GroupClustersByID(true_points)

        # ------------------------------------------------------------------
        # RECO POINTS: clustering-global (post-CLM), grouped by
        # REAL_CLUSTER_ID (not cluster_id -- see header note: cluster_id can
        # merge physically distinct tracks together, real_cluster_id is the
        # correct per-track ID, confirmed against BEE display). Cluster IDs
        # ARE further reassigned via reassign_cluster_ID_reco (relabeled by
        # avg-X, same convention as the true side) -- applied AFTER the
        # cuts, matching the original pipeline's ordering.
        # ------------------------------------------------------------------
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))

        # BEAM-WINDOW CUT -- the only thing separating this notebook from
        # Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb. Keeps the
        # clusters already identified as in-spill above (clu_beam_window_ids:
        # bridged flash time inside [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]),
        # so everything downstream -- efficiency, purity, 1-to-1 and 1-to-many
        # matching, metadata, every plot -- sees the neutrino-dominated in-spill
        # reco population only.
        #
        # Applied FIRST, before the other reco cuts and before
        # reassign_cluster_ID_reco: clu_beam_window_ids lives in
        # clustering-global's REAL_CLUSTER_ID namespace, which is column 3 here,
        # and reassign_cluster_ID_reco relabels clusters by avg-X and destroys
        # that namespace. Filtering after it would silently match nothing.
        #
        # RECO-side only, by design -- see the header cell: "in beam window" is
        # not a truth quantity, so the true side keeps its cosmic clusters and a
        # cosmic true cluster going unmatched here means "no in-spill reco
        # cluster near it".
        if Apply_beam_window_cut:
            n_reco_points_before_beam   = len(predicted_points)
            n_reco_clusters_before_beam = len(np.unique(predicted_points[:, 3])) if n_reco_points_before_beam else 0
            beam_ids_array   = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
            print(f"  Event {evt}: beam-window cut kept "
                  f"{len(clu_beam_window_ids)}/{n_reco_clusters_before_beam} reco clusters, "
                  f"{len(predicted_points)}/{n_reco_points_before_beam} reco points")

        if Apply_min_reco_points_cutoff:
            predicted_points = apply_min_reco_points_cutoff(predicted_points, min_reco_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            predicted_points = apply_wire_readout_sensitive_yz_plane_cut_reco(predicted_points, x_min, x_max, y_min, y_max, z_min, z_max)
        # An event can legitimately end up with NO in-spill reco cluster once the
        # beam-window cut is on. That event is kept rather than skipped -- its true
        # clusters are genuine reconstruction failures and belong in the efficiency
        # denominator -- but reassign_cluster_ID_reco cannot take an empty array
        # (it concatenates per-cluster blocks), so short-circuit to an empty dict.
        # EvaluateEfficiency then marks every true cluster unmatched (reco id 8888),
        # which is the correct description of the event.
        if len(predicted_points) == 0:
            clusters_reco = {}
            print(f"  Event {evt}: no reco cluster survives the beam-window cut -- "
                  f"all true clusters counted as unmatched")
        else:
            predicted_points = reassign_cluster_ID_reco(predicted_points)
            clusters_reco    = GroupClustersByID(predicted_points)

        # ------------------------------------------------------------------
        # CLUSTER CATEGORY, EFFICIENCY, PURITY (existing functions, unchanged) --
        # all computed on the clustering-level (post-CLM) clusters_true/clusters_reco above.
        # ------------------------------------------------------------------
        cluster_category_results = cluster_category(clusters_true, output_dir=None, event=evt, apa="Combined", file_name=input_file_name)

        efficiency_results = EvaluateEfficiency(clusters_true, clusters_reco, event_key, radius_efficiency, min_recopoints_threshold)
        purity_results      = EvaluatePurity(clusters_true, clusters_reco, event_key, radius_purity_xz, radius_purity_yz, radius_purity_xy)

        # ------------------------------------------------------------------
        # METADATA + 1-TO-1 / 1-TO-MANY MATCHING (existing functions, unchanged)
        # ------------------------------------------------------------------
        event_metadata_list = add_metadata_true_clusters(
            efficiency_results, cluster_category_results,
            file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)

        event_matched_pairs = MatchTrueToReco1to1(efficiency_results, purity_results)
        event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
            event_matched_pairs, cluster_category_results,
            file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)

        matched_true_reco_clusters = MatchTruetoReco_OneToMany(purity_results, efficiency_results)

        # ------------------------------------------------------------------
        # NEUTRINO/COSMIC LABEL RECORDS (event level): cluster-level
        # is_neutrino, for DrawLabelsAggregated below (see header note on
        # why DrawLabels' own strict q_true==1 check can misclassify once
        # q_true is a multi-valued neutrino index).
        #
        # NO beam-window flag is computed for true clusters. True clusters carry
        # no flash and no time, so it could only be inferred by matching to a
        # beam-window-flashed reco cluster -- mixing beam timing with
        # reconstruction efficiency while reading as a truth-level selection.
        # Beam window stays a RECO-side quantity (event_reco_beam_window_record
        # above / reco_cluster_info.txt).
        # ------------------------------------------------------------------
        event_cluster_type_records = build_true_cluster_type_records(
            clusters_true, input_file_name, evt, event_key
        )

        # ------------------------------------------------------------------
        # TRUE NEUTRINO INTERACTION VERTICES (mc.json), joined to their true
        # cluster by nu_idx (cluster_id = 99990+nu_idx -- an exact key, no
        # spatial matching). vertex_in_volume uses the wire-readout sensitive
        # box, the same bounds as the fiducial cut; the job-level scatter plot
        # is there to decide whether that is the right volume to keep.
        # Energies on these records: cluster_energy_MeV is the sed-derived true
        # cluster energy used by every cut/plot in this pipeline; mc.json's
        # Etot/Edep ride along as reference only and feed nothing.
        # ------------------------------------------------------------------
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(mc_tree), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        # ------------------------------------------------------------------
        # DRAWING (existing functions, unchanged; "Combined" used as the apa label)
        # ------------------------------------------------------------------
        if b_draw_event_level_plots:
            DrawTrueRecoClustersXZ(clusters_true, clusters_reco, evt, "Combined", PLOTDIR_EVT, input_file_name)
            DrawTrueRecoClustersYZ(clusters_true, clusters_reco, evt, "Combined", PLOTDIR_EVT, input_file_name)
            DrawTrueRecoClustersXY(clusters_true, clusters_reco, evt, "Combined", PLOTDIR_EVT, input_file_name)
            # Same three views again with the left panel restricted to the true NEUTRINO
            # clusters (right panel unchanged: all selected/beam-window reco clusters), so the
            # neutrino can be found without the cosmic tracks covering it:
            # neutrino_clusters_reco_true_event*_apa_Combined_{XZ,YZ,XY}.png
            DrawNeutrinoRecoClusters(clusters_true, clusters_reco, evt, "Combined", PLOTDIR_EVT, input_file_name)
            ##DrawLabels(clusters_true, evt, "Combined", PLOTDIR_EVT, input_file_name)
            DrawLabelsAggregated(event_cluster_type_records, event_output_dir, f"Event {evt}", f"event_{evt}", "Combined", file_name=input_file_name, vertex_records=event_vertex_records)
            DrawLabelsByNuIdx(event_cluster_type_records, event_output_dir, f"Event {evt}", f"event_{evt}", "Combined", file_name=input_file_name)
            DrawNeutrinoVertices(event_vertex_records, event_output_dir, f"Event {evt}", f"event_{evt}", "Combined",
                                 file_name=input_file_name,
                                 x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max)
            DrawNeutrinoVolumeCategory(event_vertex_records, event_output_dir, f"Event {evt}", f"event_{evt}", "Combined", file_name=input_file_name)
            DrawNeutrinoFlavor(event_vertex_records, event_output_dir, f"Event {evt}", f"event_{evt}", "Combined", file_name=input_file_name)
            DrawNeutrinoBreakdown(event_vertex_records, event_output_dir, f"Event {evt}", f"event_{evt}", "Combined", file_name=input_file_name, energy_threshold=min_cluster_energy)
            plot_efficiency_heatmap(efficiency_results, evt, "Combined", efficiency_dir, input_file_name)
            plot_purity_heatmap(purity_results, evt, "Combined", purity_dir, input_file_name)

            DrawTrueRecoMatchMultiplicity(event_metadata_list, event_output_dir, "Combined", f"Event {evt}", f"event_{evt}", file_name=input_file_name)
            for matched_info in matched_true_reco_clusters:
                DrawTrueClusterWithMatchedReco(matched_info, clusters_true, clusters_reco, efficiency_dir, evt, "Combined", input_file_name)

            DrawEfficiencyVsTrueEnergyPerEvent(efficiency_results, efficiency_2d1d_imaging_dir, evt, "Combined", input_file_name, cluster_category_results=cluster_category_results)
            DrawClusterEfficiencyVsTrueEnergyPerEvent(event_pair_metadata_list, efficiency_2d1d_clustering_dir, evt, "Combined", input_file_name, all_true_metadata_list=event_metadata_list)
            DrawEfficiencyVsTrueEnergy_MatchedPairs_PerEvent(event_pair_metadata_list, efficiency_2d1d_clustering_pairs_only_dir, evt, "Combined", input_file_name)
            DrawPurityVsRecoChargePerEvent(event_pair_metadata_list, purity_dir, evt, "Combined", input_file_name)
            DrawEfficiencyVsPurity_MatchedPairs(event_pair_metadata_list, eff_vs_purity_incl_dir, f"Event {evt}", "Combined", input_file_name, all_true_metadata_list=event_metadata_list)
            DrawEfficiencyVsPurity_MatchedPairs(event_pair_metadata_list, eff_vs_purity_excl_dir, f"Event {evt}", "Combined", input_file_name, all_true_metadata_list=None)
            plt.close('all')

        # ------------------------------------------------------------------
        # TEXT FILES FOR CONFIRMATION (event level only): dump the exact
        # underlying efficiency/purity data behind each directory's plots,
        # so cluster-by-cluster values can be checked directly instead of
        # only reading them off a plot.
        # ------------------------------------------------------------------
        # imaginglevel: same grouping DrawEfficiencyVsTrueEnergyPerEvent uses --
        # every true cluster, efficiency summed across all its reco matches.
        # clusteringlevel / clusteringlevel_true_reco_pairs_only: 1-to-1 matched pairs
        # (efficiency + purity together); the pairs-only file omits the unmatched rows
        # that the "including unmatched" table adds at efficiency 0 / matched=no.
        #
        # These are the SAME writers (writeinformation.write_efficiency_info /
        # write_pair_efficiency_info / write_purity_info) called again at file and job
        # level below -- so an event-level file is that event's slice of the aggregated
        # one, byte for byte, and the three levels can never drift apart in format. That
        # is why they carry an 'event' column even here, where it is constant.
        write_efficiency_info(event_metadata_list, efficiency_2d1d_imaging_dir)
        write_pair_efficiency_info(event_pair_metadata_list, efficiency_2d1d_clustering_dir,
                                   all_true_metadata_list=event_metadata_list)
        write_pair_efficiency_info(event_pair_metadata_list, efficiency_2d1d_clustering_pairs_only_dir)

        # true/reco cluster info: the SAME writers (writeinformation.write_true_cluster_info /
        # write_reco_cluster_info) used for job_summary/ below, fed this event's
        # records only -- so the event-level file is that event's row of the job-level
        # file, byte for byte, and the two can never drift apart in format.
        write_true_cluster_info(event_cluster_type_records, event_output_dir)
        write_reco_cluster_info([event_reco_beam_window_record], event_output_dir)

        # true_neutrino_info.txt / removed_true_neutrino_info.txt for THIS event, from the
        # same two writers the job level uses below (writeinformation.write_neutrino_vertex_info /
        # write_removed_neutrino_info), fed this event's vertex records only -- so the
        # event-level file is that event's slice of the job-level one, byte for byte.
        # The removed-neutrino file needs build_neutrino_vertex_records to have been given
        # clusters_true_precut (it was, above) to say WHY an interaction was removed.
        write_neutrino_vertex_info(event_vertex_records, event_output_dir)
        write_removed_neutrino_info(event_vertex_records, event_output_dir)

        # purity: raw EvaluatePurity output, one row per reco cluster (matched or not --
        # unmatched reco clusters carry the true_cluster_id=8888 / purity=-0.1 sentinel).
        write_purity_info(purity_results, purity_dir)

        # ------------------------------------------------------------------
        # AGGREGATE TO FILE AND JOB LEVEL
        # ------------------------------------------------------------------
        file_efficiency_results.extend(efficiency_results)
        file_purity_results.extend(purity_results)
        file_metadata_list.extend(event_metadata_list)
        file_pair_metadata_list.extend(event_pair_metadata_list)

        job_efficiency_results.extend(efficiency_results)
        job_purity_results.extend(purity_results)
        job_metadata_list.extend(event_metadata_list)
        job_pair_metadata_list.extend(event_pair_metadata_list)

        file_flash_metadata_list.extend(event_flash_metadata_list)
        job_flash_metadata_list.extend(event_flash_metadata_list)
        file_img_cluster_flash_records.extend(event_img_cluster_flash_records)
        job_img_cluster_flash_records.extend(event_img_cluster_flash_records)

        file_cluster_type_records.extend(event_cluster_type_records)
        job_cluster_type_records.extend(event_cluster_type_records)
        file_vertex_records.extend(event_vertex_records)
        job_vertex_records.extend(event_vertex_records)
        file_reco_beam_window_records.append(event_reco_beam_window_record)
        job_reco_beam_window_records.append(event_reco_beam_window_record)

        n_neutrino_points       = int(np.sum(true_points[:, 4] > 0))
        mc_records              = flatten_mc_tree(mc_tree)
        interaction_vertices    = [(r['particle'], r['energy_MeV']) for r in mc_records if r['is_interaction_vertex']]

        print(
            f"  Event {evt}: "
            f"flashes matched to clusters={len(event_flash_metadata_list)} (of {len(op_data['op_t'])} total flashes), "
            f"clusters in beam window={len(clusters_in_beam_window)}, "
            f"clustering clusters in beam window={len(clusters_clu_in_beam_window)}, "
            f"true clusters={len(clusters_true)}, reco clusters={len(clusters_reco)}, "
            f"efficiency records={len(efficiency_results)}, purity records={len(purity_results)}, 1-to-1 pairs={len(event_pair_metadata_list)}, "
            f"neutrino points={n_neutrino_points}, "
            f"interaction vertices={interaction_vertices}"
        )
        total_events_processed += 1

    # ========================================================================
    # FILE-LEVEL AGGREGATION (existing functions, unchanged)
    # ========================================================================
    # Neutrino selection tables for this FILE, same two writers again. Deliberately
    # ahead of the plotting block below and outside its guard: they describe the true
    # neutrino population, which exists whether or not this file produced any
    # efficiency/purity record.
    file_agg_output_dir = file_output_dir / "file_summary"
    write_neutrino_vertex_info(file_vertex_records, file_agg_output_dir)
    write_removed_neutrino_info(file_vertex_records, file_agg_output_dir)

    # 'or', not 'and': a file whose every event lost all its reco clusters to the
    # beam-window cut has no purity records at all, but its true clusters are still
    # real reconstruction failures that belong in the file-level efficiency plots and
    # tables. Every drawer below tolerates an empty pair/purity list.
    if file_efficiency_results or file_purity_results:
        print(f"\n  FILE-LEVEL AGGREGATION ({input_file_name}): "
              f"{len(file_efficiency_results)} efficiency, {len(file_purity_results)} purity, "
              f"{len(file_pair_metadata_list)} 1-to-1 pairs")

        agg_output_dir = file_output_dir / "file_summary"
        agg_output_dir.mkdir(parents=True, exist_ok=True)

        file_efficiency_2d1d_imaging_dir               = agg_output_dir / "efficiency" / "efficiency_2d_1d_imaginglevel"
        file_efficiency_2d1d_clustering_dir            = agg_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel"
        file_efficiency_2d1d_clustering_pairs_only_dir = agg_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
        file_efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
        file_efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
        file_efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

        file_purity_summary_dir = agg_output_dir / "purity"
        file_purity_summary_dir.mkdir(parents=True, exist_ok=True)

        file_eff_vs_purity_incl_dir = agg_output_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
        file_eff_vs_purity_excl_dir = agg_output_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
        file_eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
        file_eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

        # FILE-LEVEL TEXT TABLES: the same three writers as the event level, fed this
        # file's records. Written regardless of b_draw_file_level_plots -- these are the
        # numbers behind the plots, worth having even on a run with drawing turned off.
        write_efficiency_info(file_metadata_list, file_efficiency_2d1d_imaging_dir)
        write_pair_efficiency_info(file_pair_metadata_list, file_efficiency_2d1d_clustering_dir,
                                   all_true_metadata_list=file_metadata_list)
        write_pair_efficiency_info(file_pair_metadata_list, file_efficiency_2d1d_clustering_pairs_only_dir)
        write_purity_info(file_purity_results, file_purity_summary_dir)

        if b_draw_file_level_plots:
            DrawEfficiencyVsTrueEnergyPerFile(file_efficiency_results, file_efficiency_2d1d_imaging_dir, "Combined", input_file_name, file_metadata_list=file_metadata_list)
            DrawClusterEfficiencyVsTrueEnergyPerFile(file_pair_metadata_list, file_efficiency_2d1d_clustering_dir, "Combined", input_file_name, all_true_metadata_list=file_metadata_list)
            DrawEfficiencyVsTrueEnergy_MatchedPairs_PerFile(file_pair_metadata_list, file_efficiency_2d1d_clustering_pairs_only_dir, "Combined", input_file_name)
            DrawPurityVsRecoChargePerFile(file_pair_metadata_list, file_purity_summary_dir, "Combined", input_file_name)
            DrawEfficiencyVsPurity_MatchedPairs(file_pair_metadata_list, file_eff_vs_purity_incl_dir, "File Level", "Combined", input_file_name, all_true_metadata_list=file_metadata_list)
            DrawEfficiencyVsPurity_MatchedPairs(file_pair_metadata_list, file_eff_vs_purity_excl_dir, "File Level", "Combined", input_file_name, all_true_metadata_list=None)
            _draw_cosmic_category_bar(file_metadata_list, agg_output_dir, "Combined", "File Level", "file", file_name=input_file_name)
            DrawLabelsAggregated(file_cluster_type_records, agg_output_dir, "File Level", "file", "Combined", file_name=input_file_name, vertex_records=file_vertex_records)
            DrawLabelsByNuIdx(file_cluster_type_records, agg_output_dir, "File Level", "file", "Combined", file_name=input_file_name)
            DrawNeutrinoVertices(file_vertex_records, agg_output_dir, "File Level", "file", "Combined",
                                 file_name=input_file_name,
                                 x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max)
            DrawNeutrinoVolumeCategory(file_vertex_records, agg_output_dir, "File Level", "file", "Combined", file_name=input_file_name)
            DrawNeutrinoFlavor(file_vertex_records, agg_output_dir, "File Level", "file", "Combined", file_name=input_file_name)
            DrawNeutrinoBreakdown(file_vertex_records, agg_output_dir, "File Level", "file", "Combined", file_name=input_file_name, energy_threshold=min_cluster_energy)
            DrawTrueRecoMatchMultiplicity(file_metadata_list, agg_output_dir, "Combined", "File Level", "file", file_name=input_file_name)
            plt.close('all')

    if file_flash_metadata_list:
        print(f"  FILE-LEVEL AGGREGATION ({input_file_name}): {len(file_flash_metadata_list)} flash-matched clusters")
        file_imaging_details_flashes_dir = file_output_dir / "file_summary" / "imaging_details" / "flashes"
        file_imaging_details_flashes_dir.mkdir(parents=True, exist_ok=True)
        if b_draw_file_level_plots:
            draw_flashes(file_flash_metadata_list, file_imaging_details_flashes_dir, "Combined", "File Level", "file", file_name=input_file_name)
            plt.close('all')

    if file_img_cluster_flash_records:
        file_clustering_details_flashes_dir = file_output_dir / "file_summary" / "clustering_details" / "flashes"
        file_clustering_details_flashes_dir.mkdir(parents=True, exist_ok=True)
        if b_draw_file_level_plots:
            draw_clustering_flashes(file_img_cluster_flash_records, file_clustering_details_flashes_dir, "Combined", "File Level", "file", file_name=input_file_name)
            plt.close('all')

# ============================================================================
# JOB-LEVEL AGGREGATION (existing functions, unchanged)
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total efficiency results: {len(job_efficiency_results)}")
print(f"Total purity results: {len(job_purity_results)}")
print(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
print(f"Total flash-matched clusters: {len(job_flash_metadata_list)}")
print(f"{'='*70}")

job_output_dir = output_dir / "job_summary"
job_output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# TRUE CLUSTER INFO TEXT FILE (job level; same file also written per event
# above, via the same writeinformation.write_true_cluster_info): one row per event listing how
# many neutrinos it has, and how many of those are in the beam window
# (build_true_cluster_type_records, filtered to is_neutrino;
# reassign_cluster_ID_true_charge_light keeps each neutrino interaction as
# its own true cluster -- 99990+nu_idx -- so counting neutrino clusters per
# event IS counting neutrinos per event, no need to print nu_idx_values or
# repeat a row per interaction). No beam-window column: that is a reco-side
# quantity only -- see build_true_cluster_type_records.
# ============================================================================
true_cluster_info_path = write_true_cluster_info(job_cluster_type_records, job_output_dir)
if true_cluster_info_path:
    print(f"True cluster info written to: {true_cluster_info_path}")

# ============================================================================
# RECO CLUSTER INFO TEXT FILE (job level; same file also written per event
# above, via the same writeinformation.write_reco_cluster_info): one row per event listing how many
# DISTINCT clustering-global clusters have a beam-window-matched flash
# (event_reco_beam_window_record, built above from clusters_clu_in_beam_window).
# This is the reco-side proxy for "multiple neutrino-like activity in the beam
# spill" -- grouped by reco cluster + matched flash timing, NOT by true nu_idx
# (that's true_cluster_info.txt's num_neutrinos column above, ground truth).
# ============================================================================
reco_cluster_info_path = write_reco_cluster_info(job_reco_beam_window_records, job_output_dir)
if reco_cluster_info_path:
    print(f"Reco cluster info written to: {reco_cluster_info_path}")

# ============================================================================
# TRUE NEUTRINO SELECTION OUTPUTS (job level): the same six drawers and two
# writers already run at event and file level above, now aggregated over every
# file and event in the job:
#   true_neutrino_info.txt, removed_true_neutrino_info.txt,
#   labels_aggregated_job_*, labels_by_nu_idx_job_*, true_neutrino_vertices_job_*,
#   true_neutrino_vertex_volume_job_*, true_neutrino_flavor_in_volume_job_*,
#   true_neutrino_breakdown_job_*
#
# SelectionAnalysis.ipynb writes the same filenames into its own job_summary/ from
# the same functions. That duplication is intended, not an oversight: these describe
# the true population that THIS notebook's efficiency and purity numbers were
# computed on, so they belong beside them instead of in another notebook's output
# tree. Both notebooks run the same cuts on the same input, so the two copies agree;
# if they ever disagree, the cut configurations have drifted apart and that is worth
# knowing.
#
# No file_name argument at this level -- a job spans every file, so there is no one
# file to name (the drawers put "Job Level" in the title instead).
# ============================================================================
if b_draw_job_level_plots:
    DrawLabelsAggregated(job_cluster_type_records, job_output_dir, "Job Level", "job", "Combined", vertex_records=job_vertex_records)
    DrawLabelsByNuIdx(job_cluster_type_records, job_output_dir, "Job Level", "job", "Combined")
    DrawNeutrinoVertices(job_vertex_records, job_output_dir, "Job Level", "job", "Combined",
                         x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max)
    DrawNeutrinoVolumeCategory(job_vertex_records, job_output_dir, "Job Level", "job", "Combined")
    DrawNeutrinoFlavor(job_vertex_records, job_output_dir, "Job Level", "job", "Combined")
    DrawNeutrinoBreakdown(job_vertex_records, job_output_dir, "Job Level", "job", "Combined", energy_threshold=min_cluster_energy)
    plt.close('all')

# Written regardless of b_draw_job_level_plots: these are text tables, not plots,
# and they are the record of which true neutrino interactions survived the cuts and
# which did not -- worth having even on a run with drawing turned off.
true_neutrino_info_path = write_neutrino_vertex_info(job_vertex_records, job_output_dir)
if true_neutrino_info_path:
    print(f"True neutrino info written to: {true_neutrino_info_path}")

removed_true_neutrino_info_path = write_removed_neutrino_info(job_vertex_records, job_output_dir)
if removed_true_neutrino_info_path:
    print(f"Removed true neutrino info written to: {removed_true_neutrino_info_path}")

# The EVENT- and FILE-level copies of these plots stay where they are, for debugging
# individual events and files. cosmic_clusters_by_category_* likewise runs at all
# three levels: it classifies cosmic track geometry via cluster_category, which is
# cluster categorisation rather than neutrino selection.
# ============================================================================

# 'or', not 'and': see the file-level note -- a job in which no reco cluster anywhere
# survived the beam-window cut still has true clusters to report, all unmatched.
if job_efficiency_results or job_purity_results:
    job_efficiency_2d1d_imaging_dir               = job_output_dir / "efficiency" / "efficiency_2d_1d_imaginglevel"
    job_efficiency_2d1d_clustering_dir            = job_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel"
    job_efficiency_2d1d_clustering_pairs_only_dir = job_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
    job_efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
    job_efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
    job_efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

    job_purity_summary_dir = job_output_dir / "purity"
    job_purity_summary_dir.mkdir(parents=True, exist_ok=True)

    job_eff_vs_purity_incl_dir = job_output_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
    job_eff_vs_purity_excl_dir = job_output_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
    job_eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
    job_eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

    # JOB-LEVEL TEXT TABLES: same three writers again, over every file and event in the
    # job. Written regardless of b_draw_job_level_plots, same reasoning as file level.
    write_efficiency_info(job_metadata_list, job_efficiency_2d1d_imaging_dir)
    write_pair_efficiency_info(job_pair_metadata_list, job_efficiency_2d1d_clustering_dir,
                               all_true_metadata_list=job_metadata_list)
    write_pair_efficiency_info(job_pair_metadata_list, job_efficiency_2d1d_clustering_pairs_only_dir)
    write_purity_info(job_purity_results, job_purity_summary_dir)

    if b_draw_job_level_plots:
        DrawEfficiencyVsTrueEnergyPerJob(job_efficiency_results, job_efficiency_2d1d_imaging_dir, "Combined", job_metadata_list=job_metadata_list)
        DrawClusterEfficiencyVsTrueEnergyPerJob(job_pair_metadata_list, job_efficiency_2d1d_clustering_dir, "Combined", all_true_metadata_list=job_metadata_list)
        DrawEfficiencyVsTrueEnergy_MatchedPairs_PerJob(job_pair_metadata_list, job_efficiency_2d1d_clustering_pairs_only_dir, "Combined")
        DrawPurityVsRecoChargePerJob(job_pair_metadata_list, job_purity_summary_dir, "Combined")
        DrawEfficiencyVsPurity_MatchedPairs(job_pair_metadata_list, job_eff_vs_purity_incl_dir, "Job Level", "Combined", all_true_metadata_list=job_metadata_list)
        DrawEfficiencyVsPurity_MatchedPairs(job_pair_metadata_list, job_eff_vs_purity_excl_dir, "Job Level", "Combined", all_true_metadata_list=None)
        # The six neutrino selection drawers that used to follow this line are in
        # SelectionAnalysis.ipynb now -- see the MOVED OUT note above. The cosmic
        # category bar stays: it is cluster categorisation, not neutrino selection.
        _draw_cosmic_category_bar(job_metadata_list, job_output_dir, "Combined", "Job Level", "job")
        DrawTrueRecoMatchMultiplicity(job_metadata_list, job_output_dir, "Combined", "Job Level", "job")

if job_flash_metadata_list:
    job_imaging_details_flashes_dir = job_output_dir / "imaging_details" / "flashes"
    job_imaging_details_flashes_dir.mkdir(parents=True, exist_ok=True)
    if b_draw_job_level_plots:
        draw_flashes(job_flash_metadata_list, job_imaging_details_flashes_dir, "Combined", "Job Level", "job")

if job_img_cluster_flash_records:
    job_clustering_details_flashes_dir = job_output_dir / "clustering_details" / "flashes"
    job_clustering_details_flashes_dir.mkdir(parents=True, exist_ok=True)
    if b_draw_job_level_plots:
        draw_clustering_flashes(job_img_cluster_flash_records, job_clustering_details_flashes_dir, "Combined", "Job Level", "job")

plt.close('all')

# ============================================================================
# JOB SUMMARY TEXT FILE
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime = time.time() - job_start_time

n_cathode_crossing  = sum(1 for r in job_img_cluster_flash_records if r['is_cathode_crossing'])
n_img_beam_window   = sum(1 for r in job_flash_metadata_list if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US)
n_clu_beam_window   = sum(1 for r in job_img_cluster_flash_records if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US)

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
if target_file is not None or target_event is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("CLUSTERING-LEVEL evaluation (after charge-light-matching, CLM) is active:")
summary_lines.append("selections / cluster_category / efficiency / purity / 1-to-1 and 1-to-many")
summary_lines.append("matching / metadata / drawing all computed on sed-smear_readout (true) vs")
summary_lines.append("clustering-global (reco), plus optical-flash processing (img-global <->")
summary_lines.append("clustering-global flash bridging).")
summary_lines.append("")
summary_lines.append("Beam Window / Cathode-Crossing Parameters:")
summary_lines.append(f"  beam_window_min_us: {BEAM_WINDOW_MIN_US}")
summary_lines.append(f"  beam_window_max_us: {BEAM_WINDOW_MAX_US}")
summary_lines.append(f"  cathode_crossing_time_diff_max_us: {CATHODE_CROSSING_TIME_DIFF_MAX_US}")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
summary_lines.append(f"Total efficiency results: {len(job_efficiency_results)}")
summary_lines.append(f"Total purity results: {len(job_purity_results)}")
summary_lines.append(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
summary_lines.append(f"Total img-global flash-matched cluster records: {len(job_flash_metadata_list)}")
summary_lines.append(f"Total clustering-global flash-matched cluster records: {len(job_img_cluster_flash_records)}")
summary_lines.append(f"Total cathode-crossing merges detected: {n_cathode_crossing}")
summary_lines.append(f"Total img-global cluster records in beam window: {n_img_beam_window}")
summary_lines.append(f"Total clustering-global cluster records in beam window: {n_clu_beam_window}")
summary_lines.append("")

if job_flash_metadata_list:
    all_img_times = [r['flash_time'] for r in job_flash_metadata_list]
    summary_lines.append("Flash Time Statistics (img-global level, per matched cluster record):")
    summary_lines.append(f"  Mean flash time:   {np.mean(all_img_times):.4f} us")
    summary_lines.append(f"  Median flash time: {np.median(all_img_times):.4f} us")
    summary_lines.append(f"  Min flash time:    {np.min(all_img_times):.4f} us")
    summary_lines.append(f"  Max flash time:    {np.max(all_img_times):.4f} us")
    summary_lines.append("")

if job_img_cluster_flash_records:
    all_clu_times = [r['flash_time'] for r in job_img_cluster_flash_records]
    summary_lines.append("Flash Time Statistics (clustering-global level, per matched cluster record):")
    summary_lines.append(f"  Mean flash time:   {np.mean(all_clu_times):.4f} us")
    summary_lines.append(f"  Median flash time: {np.median(all_clu_times):.4f} us")
    summary_lines.append(f"  Min flash time:    {np.min(all_clu_times):.4f} us")
    summary_lines.append(f"  Max flash time:    {np.max(all_clu_times):.4f} us")
    summary_lines.append("")

# Efficiency performance behind the efficiency_2d_1d_clusteringlevel 1D plots:
# mean efficiency below vs above 500 MeV, over the same population those plots
# use (1-to-1 pairs + unmatched true clusters at efficiency=0 -- the same two
# lists passed to DrawClusterEfficiencyVsTrueEnergyPerJob above).
efficiency_energy_records = summarize_cluster_efficiency_by_energy(
    job_pair_metadata_list, all_true_metadata_list=job_metadata_list, energy_threshold=500)
summary_lines.extend(format_cluster_efficiency_by_energy(efficiency_energy_records, energy_threshold=500))

summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append("=" * 80)

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")



Output directory: multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut/combined_apa_20260803_143812


FILE 1/12: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Processing events 0 to 9

  Event 0: beam-window cut kept 0/15 reco clusters, 0/21848 reco points
  Event 0: no reco cluster survives the beam-window cut -- all true clusters counted as unmatched

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 0: flashes matched to clusters=14 (of 50 total flashes), clusters in beam window=0, clustering clusters in beam window=0, true clusters=7, reco clusters=0, efficiency records=7, purity records=0, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 49.8)]
  Event 1: beam-window cut kept 1/13 reco clusters, 740/18105 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
Found 1 true clusters with matched reco clusters (1-to-many)

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 109: flashes matched to clusters=9 (of 32 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=5, reco clusters=1, efficiency records=5, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 0.0), ('numu', 7.8)]

  FILE-LEVEL AGGREGATION (file10): 101 efficiency, 6 purity, 4 1-to-1 pairs
    [FILE LEVEL] Built category_info_source from metadata: 101 clusters
    [FILE LEVEL] Drawing 2D efficiency plots for categories...
      Neutrino Clusters: found 4 clusters in metadata
        → Matched 4 clusters in efficiency_results
        → Drawing 2D plot for Neutrino Clusters...
      Isochronous Cosmic Clusters: found 17 clusters in metadata
        → Matched 17 clusters in efficiency_results
        → Drawing 2D plot for Isochronous Cosmic Clusters...
      Normal Cosmic Clusters: found 66 clusters in metadata
        → Matched 66 clusters in efficiency_results
        → Drawing 2D plot for Normal Cosmic Cluste

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 110: flashes matched to clusters=10 (of 34 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=7, reco clusters=1, efficiency records=7, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 0.0), ('numu', 14.8), ('numu', 0.0), ('numu', 0.0)]
  Event 111: beam-window cut kept 1/13 reco clusters, 3050/23839 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
Found 1 true clusters with matched reco clusters (1-to-many)
  Event 111: flashes matched to clusters=10 (of 27 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=7, reco clusters=1, efficiency records=7, purity records=1, 1-to-1 pairs=1, neutrino points=6852, interaction vertices=[('numu', 0.0), ('numu', 438.0)]
  Event 112: beam-window cut kept 0/9 reco clusters, 0/36943 reco points
  Event 112: no reco cluster survives the beam-window cut -- all true clusters counted as unmatched

Found 0 

...

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 66: flashes matched to clusters=16 (of 37 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=11, reco clusters=1, efficiency records=11, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 37.5)]
  Event 67: beam-window cut kept 1/13 reco clusters, 1864/22983 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
Found 1 true clusters with matched reco clusters (1-to-many)
  Event 67: flashes matched to clusters=12 (of 31 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=10, reco clusters=1, efficiency records=10, purity records=1, 1-to-1 pairs=1, neutrino points=7553, interaction vertices=[('numu', 581.8)]
  Event 68: beam-window cut kept 0/18 reco clusters, 0/33795 reco points
  Event 68: no reco cluster survives the beam-wind

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 83: flashes matched to clusters=16 (of 39 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=8, reco clusters=1, efficiency records=8, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 48.2)]
  Event 84: beam-window cut kept 0/12 reco clusters, 0/36107 reco points
  Event 84: no reco cluster survives the beam-window cut -- all true clusters counted as unmatched

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 84: flashes matched to clusters=14 (of 34 total flashes), clusters in beam window=0, clustering clusters in beam window=0, true clusters=9, reco clusters=0, efficiency records=9, purity records=0, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 5.5)]
  Event 85: beam-window cut kept 3/19 reco clusters, 5030/27999 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
Found 1 true cluster

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 92: flashes matched to clusters=14 (of 36 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=8, reco clusters=1, efficiency records=8, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 81.7), ('numu', 26.6), ('numu', 0.0)]
  Event 93: beam-window cut kept 0/17 reco clusters, 0/51567 reco points
  Event 93: no reco cluster survives the beam-window cut -- all true clusters counted as unmatched

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 93: flashes matched to clusters=15 (of 35 total flashes), clusters in beam window=0, clustering clusters in beam window=0, true clusters=11, reco clusters=0, efficiency records=11, purity records=0, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 69.5), ('numu', 0.0)]
  Event 94: beam-window cut kept 1/9 reco clusters, 544/25110 reco points

Found 1 matched pairs of true 